# Used Vehicle Sales & Pricing Analytics

Run this notebook from top to bottom. It loads the raw CSV, runs the project notebooks in order in a shared Python session, displays their outputs, and creates the visualizations at the end. Keep this file in the same folder as the other notebooks and `Raw_data.csv`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Find the project folder whether Jupyter starts in this folder or its parent.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "Raw_data.csv").exists() and (PROJECT_DIR / "Python" / "Raw_data.csv").exists():
    PROJECT_DIR = PROJECT_DIR / "Python"

required_files = [
    "Raw_data.csv",
    "1 Data_Quality_Checking.ipynb",
    "2 Clean_Make.ipynb",
    "3 Clean_Body.ipynb",
    "4 Clean_color_and_Interior.ipynb",
    "5 Data Quality columns.ipynb",
    "6 Feature engineering.ipynb",
    "7 Duplicate analysis.ipynb",
    "8 SaleDate.ipynb",
]
missing_files = [name for name in required_files if not (PROJECT_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError("Missing project files in " + str(PROJECT_DIR) + ": " + ", ".join(missing_files))

print("Project folder:", PROJECT_DIR)
df = pd.read_csv(PROJECT_DIR / "Raw_data.csv", low_memory=False)
df.columns = df.columns.str.strip().str.lower()
print("Raw data loaded:", f"{len(df):,} rows × {len(df.columns)} columns")
display(df.head())


In [ ]:
# Execute each notebook's code cells in this notebook's shared namespace.
# This lets every transformation update the same DataFrame named df.
import nbformat
from IPython import get_ipython

pipeline_notebooks = [
    "1 Data_Quality_Checking.ipynb",
    "2 Clean_Make.ipynb",
    "3 Clean_Body.ipynb",
    "4 Clean_color_and_Interior.ipynb",
    "5 Data Quality columns.ipynb",
    "6 Feature engineering.ipynb",
    "7 Duplicate analysis.ipynb",
    "8 SaleDate.ipynb",
]

ip = get_ipython()
for notebook_name in pipeline_notebooks:
    print("\n" + "=" * 72)
    print("RUNNING:", notebook_name)
    print("=" * 72)
    pipeline_nb = nbformat.read(PROJECT_DIR / notebook_name, as_version=4)
    for cell_number, cell in enumerate(pipeline_nb.cells, start=1):
        if cell.cell_type != "code" or not cell.source.strip():
            continue
        # These notebooks are expected to share the same df. The first quality
        # notebook temporarily title-cases column names, so normalize them after it.
        result = ip.run_cell(cell.source, store_history=False)
        if result.error_before_exec is not None or result.error_in_exec is not None:
            raise RuntimeError(
                f"Failed in {notebook_name}, code cell {cell_number}. "
                "See the traceback above; fix that source notebook cell and rerun."
            )
    if notebook_name == "1 Data_Quality_Checking.ipynb":
        df.columns = [str(col).strip().lower() for col in df.columns]
    print("Completed:", notebook_name)

print("\nPIPELINE COMPLETE")
print("Final dataset:", f"{len(df):,} rows × {len(df.columns)} columns")
print("Final columns:", df.columns.tolist())
display(df.head())


## Final visualizations

The charts below use the final transformed `df` and match the column names created by this project. Scatter plots are sampled for performance on the large dataset.

In [ ]:
# ============================================================
# USED VEHICLE SALES & PRICING ANALYTICS — FINAL VISUALIZATIONS
# ============================================================
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
data = df.copy()

# Normalize numeric columns safely for plotting.
for col in ["sellingprice", "mmr", "odometer", "condition", "vehicle age", "year"]:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce")

# Helper: show only charts whose required columns exist and contain usable data.
def show_chart(title):
    plt.title(title)
    plt.tight_layout()
    plt.show()

# 1. Top 10 vehicle makes
make_col = "make_clean" if "make_clean" in data.columns else "make"
if make_col in data.columns:
    counts = data[make_col].dropna().astype(str).value_counts().head(10).sort_values()
    if not counts.empty:
        counts.plot(kind="barh")
        plt.xlabel("Number of vehicles")
        plt.ylabel("Vehicle make")
        show_chart("Top 10 Makes by Vehicles Sold")

# 2. Vehicle age vs selling price
if {"vehicle age", "sellingprice"}.issubset(data.columns):
    p = data[["vehicle age", "sellingprice"]].dropna()
    if len(p) > 15000: p = p.sample(15000, random_state=42)
    if not p.empty:
        sns.scatterplot(data=p, x="vehicle age", y="sellingprice", alpha=.35, s=18)
        show_chart("Vehicle Age vs Selling Price")

# 3. Mileage vs selling price
if {"odometer", "sellingprice"}.issubset(data.columns):
    p = data[["odometer", "sellingprice"]].dropna()
    if len(p) > 15000: p = p.sample(15000, random_state=42)
    if not p.empty:
        sns.scatterplot(data=p, x="odometer", y="sellingprice", alpha=.35, s=18)
        show_chart("Mileage vs Selling Price")

# 4. Condition vs average selling price
if {"condition", "sellingprice"}.issubset(data.columns):
    p = data.groupby("condition", observed=True)["sellingprice"].mean().dropna().reset_index()
    if not p.empty:
        sns.lineplot(data=p, x="condition", y="sellingprice", marker="o")
        show_chart("Condition vs Average Selling Price")

# 5. Average selling price by mileage band
if {"mileage band", "sellingprice"}.issubset(data.columns):
    p = data.groupby("mileage band", observed=True)["sellingprice"].mean().dropna().reset_index()
    if not p.empty:
        sns.barplot(data=p, x="mileage band", y="sellingprice")
        plt.xticks(rotation=30)
        show_chart("Average Selling Price by Mileage Band")

# 6. Vehicles sold by transmission
if "transmission" in data.columns:
    p = data["transmission"].fillna("Unknown").astype(str).value_counts().head(10).reset_index()
    p.columns = ["transmission", "vehicle_count"]
    if not p.empty:
        sns.barplot(data=p, x="transmission", y="vehicle_count")
        plt.xticks(rotation=30)
        show_chart("Vehicles Sold by Transmission")

# 7. MMR vs actual selling price
if {"mmr", "sellingprice"}.issubset(data.columns):
    p = data[["mmr", "sellingprice"]].dropna()
    if len(p) > 15000: p = p.sample(15000, random_state=42)
    if not p.empty:
        sns.scatterplot(data=p, x="mmr", y="sellingprice", alpha=.35, s=18)
        lo = min(p["mmr"].min(), p["sellingprice"].min())
        hi = max(p["mmr"].max(), p["sellingprice"].max())
        plt.plot([lo, hi], [lo, hi], linestyle="--", color="red", label="Selling Price = MMR")
        plt.legend()
        show_chart("MMR vs Actual Selling Price")

# 8. Top 10 states by sales
if "state" in data.columns:
    p = data["state"].dropna().astype(str).value_counts().head(10)
    if not p.empty:
        sns.barplot(x=p.values, y=p.index)
        plt.xlabel("Number of vehicles")
        plt.ylabel("State")
        show_chart("Top 10 States by Vehicle Sales")

# 9. Top 10 body types
body_col = "body_clean" if "body_clean" in data.columns else "body"
if body_col in data.columns:
    p = data[body_col].dropna().astype(str).value_counts().head(10)
    if not p.empty:
        sns.barplot(x=p.values, y=p.index)
        plt.xlabel("Number of vehicles")
        plt.ylabel("Body type")
        show_chart("Top 10 Vehicle Body Types by Sales")

# 10. Pricing status distribution
if "pricing status" in data.columns:
    p = data["pricing status"].fillna("Unknown").value_counts()
    if not p.empty:
        plt.figure(figsize=(8, 6))
        plt.pie(p.values, labels=p.index, autopct="%1.1f%%", startangle=90)
        show_chart("Vehicles Sold Above, Below or At MMR")

# 11. Selling price distribution
if "sellingprice" in data.columns:
    p = data["sellingprice"].dropna()
    if not p.empty:
        sns.histplot(p, bins=50, kde=True)
        plt.xlabel("Selling price")
        plt.ylabel("Frequency")
        show_chart("Distribution of Vehicle Selling Prices")

# 12. Selling price box plot
if "sellingprice" in data.columns:
    p = data["sellingprice"].dropna()
    if not p.empty:
        sns.boxplot(x=p)
        plt.xlabel("Selling price")
        show_chart("Selling Price Spread and Potential Outliers")

# 13. Correlation heatmap
corr_cols = [c for c in ["sellingprice", "mmr", "odometer", "vehicle age", "condition"] if c in data.columns]
if len(corr_cols) >= 2:
    corr = data[corr_cols].corr(numeric_only=True)
    if not corr.empty:
        sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
        show_chart("Correlation Between Vehicle Characteristics")

# 14. K-Means cluster chart (only if a Cluster column already exists)
cluster_col = next((c for c in data.columns if c.lower() == "cluster"), None)
if cluster_col and {"odometer", "sellingprice"}.issubset(data.columns):
    p = data[["odometer", "sellingprice", cluster_col]].dropna()
    if len(p) > 15000: p = p.sample(15000, random_state=42)
    if not p.empty:
        sns.scatterplot(data=p, x="odometer", y="sellingprice", hue=cluster_col, palette="tab10", alpha=.6)
        show_chart("K-Means Vehicle Segmentation")
else:
    print("K-Means chart skipped: no Cluster column exists in the final dataset.")

print("All available visualizations completed.")
